# SC4001 Phase 5: a frozen ESM-2 protein branch on cold-target splits

Accelerator must be **GPU (T4 x2 or P100)** and **internet must be ON**: the ESM-2 weights are
downloaded from Hugging Face in cell 3. Code and processed tables come from the attached
dataset `isaacwjk21/sc4001-project`.

## Why this kernel exists

Phase 3 is settled and negative. With an adversary that demonstrably trains and compute held
equal, DANN does not improve cold-target generalisation on DAVIS (test MSE +0.0038, t(2)=+0.28)
and is significantly worse on KIBA (+0.0728, t(2)=+8.92, p=0.012; CI -0.0226, p=0.036, all
three seed signs negative).

The surviving finding from that work is not about DANN at all. **Before any domain adaptation,
the baseline already under-disperses its predictions across unseen targets**: the mean
within-ligand prediction spread is 0.576 of the measured spread on DAVIS and 0.514 on KIBA, on
a matched ligand set. Every DANN arm then moved MSE and spread together (r = -0.951 over five
arms). That says the cold-target deficit is **representational** -- the character CNN does not
extract enough from an unseen kinase sequence to separate it from other unseen kinases -- and
that aligning feature marginals cannot create information the protein encoder never had.

This kernel tests that reading directly, by changing only the protein representation:

* **Protein branch:** `ProteinEncoder` (embedding + 3 valid Conv1d over 1200/1000 characters,
  global max-pool) is replaced by `EsmProteinEncoder`: a **frozen** mean-pooled
  `facebook/esm2_t33_650M_UR50D` embedding per target, 1280-d, projected by one
  `Dropout -> Linear(1280, 96) -> ReLU`.
* **Everything else is identical:** same drug CNN, same 96+96 fusion width, same
  1024/1024/512 head, same Adam 1e-3, batch 256, same splits, same seeds, same epoch budget.
  The projection width is `config.filters[-1]`, so the regression head is byte-identical in
  shape to the CNN arm and the protein representation is the only variable.

The language model is never fine-tuned, so its embedding is a fixed feature per sequence. It is
computed once in cell 3 rather than inside every run, which keeps the weekly GPU budget on
training.

## Comparison arms, already measured (do not retrain them)

| arm | epochs | test MSE (all) | test CI (all) | test MSE (uncens.) | test CI (uncens.) | within-ligand CI | spread/measured |
| --- | --- | --- | --- | --- | --- | --- | --- |
| DAVIS CNN baseline | 100, no early stop | 0.3991 (sd 0.0216) | 0.7884 | 1.1237 | 0.6502 | 0.6789 | 0.576 |
| KIBA CNN baseline | 80, no early stop | 0.3501 (sd 0.0281) | 0.7715 | - | - | 0.6363 | 0.514 |
| DAVIS k-mer kNN | none (not learned) | 0.3516 | 0.8385 | 0.6875 | 0.7138 | 0.7494 | 0.771 |
| KIBA k-mer kNN | none (not learned) | 0.2788 | 0.7871 | - | - | 0.6467 | 0.401 |

The k-mer row is a non-neural control run on CPU in the same session: for an unseen target it
predicts a similarity-weighted average of that drug's own training scores. It already beats the
CNN on both datasets, which is the first independent sign that the deficit is representational
rather than a tuning or adaptation problem.

## Budget, and what it costs if this is wrong

The weekly Kaggle GPU allowance left for this session is about **1.5 h**. The ESM arm drops the
protein CNN, which was the dominant cost per epoch, so each run should be cheaper than its CNN
counterpart (DAVIS CNN 13.2 min/run, KIBA CNN 37.7 min/run) -- but that is a projection, not a
measurement. So cell 4 holds a wall-clock guard: it stops launching new runs once the next run's
projected cost would push the grid past **55 min**, and reports what completed. DAVIS runs
first, so if the guard fires it is KIBA seeds that are dropped, and the DAVIS answer survives
intact. Per-run cost is printed as s/epoch so the projection can be checked against reality.

## What would make this a negative result

* ESM matches or loses to the CNN on test MSE at equal compute. Then the deficit is not
  (only) the protein representation, and the Phase 3 reading needs revising.
* ESM improves MSE but **not** within-ligand CI or prediction spread. That would mean it bought
  calibration, not the ability to separate unseen targets, and the pooled numbers would be
  hiding the same failure the CNN has.
* A 1280-d embedding squeezed through `Linear(1280, 96)` could bottleneck. If ESM loses, that
  width is the first thing to vary before concluding anything about protein language models.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())
KERNEL_STARTED = time.time()


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# This kernel needs the two modules added for Phase 5 and the extra split columns.
required = ["esm_embed.py", "kmer_baseline.py"]
present = {p.name for p in (project / "src").iterdir()}
missing = [name for name in required if name not in present]
if missing:
    raise SystemExit("Attached dataset predates Phase 5; missing src modules: " + str(missing)
                     + ". Push a new dataset version before running this kernel.")
print("phase 5 modules OK")


In [ ]:
# Frozen ESM-2 embeddings, computed once on the GPU. 766 sequences in total; on CPU this took
# 8.64 s/sequence locally (110 min), which is why it runs here instead.
import transformers
print("transformers", transformers.__version__, flush=True)

t0 = time.time()
run = subprocess.run(
    [sys.executable, "src/esm_embed.py", "--sources", "davis", "kiba", "external"],
    cwd=project, env=env, check=False,
)
print("embed rc =", run.returncode, "wall =", format((time.time() - t0) / 60, ".1f"), "min",
      flush=True)
if run.returncode != 0:
    raise SystemExit("ESM-2 embedding failed. With internet off the weights cannot be "
                     "downloaded: check the kernel's internet setting before anything else.")

# Verify every embedding file against its processed target table before training reads it.
import numpy as np
import pandas as pd

processed = project / "data" / "processed"
EXPECTED = {"davis": "davis_targets", "kiba": "kiba_targets", "external": "external_kinases"}
for source, table_stem in EXPECTED.items():
    path = processed / (source + "_esm.npz")
    if not path.exists():
        raise SystemExit("Missing " + str(path))
    with np.load(path, allow_pickle=False) as arrays:
        embeddings = arrays["prot_embeddings"]
    stem = processed / (table_stem + ".csv")
    table = pd.read_csv(stem if stem.exists() else stem.with_suffix(".csv.gz"), usecols=[0])
    if embeddings.shape != (len(table), 1280):
        raise SystemExit(source + " embeddings are " + str(embeddings.shape)
                         + ", expected " + str((len(table), 1280)))
    if not np.isfinite(embeddings).all():
        raise SystemExit(source + " embeddings contain a nonfinite value.")
    norms = np.linalg.norm(embeddings, axis=1)
    duplicates = len(table) - len(np.unique(embeddings, axis=0))
    print("%-9s %s  L2 norm min %.3f mean %.3f max %.3f  duplicate rows %d"
          % (source, embeddings.shape, norms.min(), norms.mean(), norms.max(), duplicates),
          flush=True)
    if duplicates:
        print("   NOTE: identical embeddings mean two targets are indistinguishable to the "
              "protein branch. DAVIS has 423 distinct sequences, so this should be 0.",
              flush=True)


In [ ]:
# Frozen-ESM baseline on the cold-target split, three training seeds per dataset.
#
# Epoch budgets match the CNN comparison arms exactly: DAVIS 100 epochs with no early stopping
# (its CNN baseline peaked as late as epoch 93), KIBA 80 (every KIBA baseline peaked by 74).
# Both use --patience equal to --max-epochs, so the stopping rule cannot decide the result.
#
# DAVIS runs first. If the wall-clock guard fires it drops KIBA seeds, which leaves the DAVIS
# answer complete rather than two half-answers.
ROOT = "/kaggle/working/outputs/esm"
GRID = [("davis", seed, 100) for seed in (4001, 4002, 4003)]
GRID += [("kiba", seed, 80) for seed in (4001, 4002, 4003)]

# Measured CNN cost is 13.2 min (DAVIS) and 37.7 min (KIBA) per run. The ESM arm has no protein
# CNN, so it should be cheaper, but until one run finishes that is a guess: start from the CNN
# figure so the first projection is conservative rather than optimistic.
PRIOR_MINUTES = {"davis": 13.2, "kiba": 37.7}
BUDGET_SECONDS = 55 * 60

status, skipped = [], []
for dataset, seed, epochs in GRID:
    elapsed = time.time() - KERNEL_STARTED
    same = [row[3] for row in status if row[0] == dataset and row[2] == 0]
    projected = max(same) if same else PRIOR_MINUTES[dataset] * 60
    if elapsed + projected > BUDGET_SECONDS:
        skipped.append((dataset, seed, round(projected / 60, 1)))
        print("SKIPPING " + dataset + " seed" + str(seed) + ": projected "
              + format(projected / 60, ".1f") + " min would exceed the "
              + format(BUDGET_SECONDS / 60, ".0f") + " min guard ("
              + format(elapsed / 60, ".1f") + " min used)", flush=True)
        continue
    print()
    print("===== esm " + dataset + " split_cold_target seed" + str(seed) + " ("
          + str(epochs) + " epochs, no early stop) =====", flush=True)
    t0 = time.time()
    run = subprocess.run(
        [sys.executable, "src/train.py", "--dataset", dataset, "--split", "split_cold_target",
         "--seed", str(seed), "--protein-encoder", "esm",
         "--max-epochs", str(epochs), "--patience", str(epochs), "--resume"],
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=ROOT), check=False,
    )
    wall = time.time() - t0
    status.append((dataset, seed, run.returncode, wall, epochs))
    print("----- rc=" + str(run.returncode) + " wall=" + format(wall / 60, ".1f") + " min ("
          + format(wall / epochs, ".2f") + " s/epoch) -----", flush=True)

print()
print("Grid wall time", format((time.time() - KERNEL_STARTED) / 60, ".1f"), "min since kernel start")
for dataset, seed, rc, wall, epochs in status:
    print("  %-6s seed%d rc=%d %6.1f min  %5.2f s/epoch" % (dataset, seed, rc, wall / 60, wall / epochs))
failed = [row for row in status if row[2] != 0]
print("FAILED RUNS:", failed if failed else "none")
print("SKIPPED FOR BUDGET:", skipped if skipped else "none")

# Speed-up against the CNN arm, which is the thing the projection above guessed at.
for dataset in ("davis", "kiba"):
    rates = [w / e for d, s, rc, w, e in status if d == dataset and rc == 0]
    if rates:
        cnn = PRIOR_MINUTES[dataset] * 60 / (100 if dataset == "davis" else 80)
        print("  %-6s esm %.2f s/epoch vs cnn %.2f s/epoch = %.2fx"
              % (dataset, sum(rates) / len(rates), cnn, cnn / (sum(rates) / len(rates))))


In [ ]:
# Test metrics at each run's own best checkpoint, against the already-measured CNN baseline.
import statistics as st

CNN = {
    "davis": {"all": {"mse": [0.3991, 0.4208, 0.3775], "ci": [0.7901, 0.7879, 0.7874]},
              "uncensored": {"mse": [1.2087, 1.1723, 0.9901], "ci": [0.6454, 0.6387, 0.6665]}},
    "kiba": {"all": {"mse": [0.3817, 0.3409, 0.3278], "ci": [0.7662, 0.7697, 0.7785]}},
}
KMER = {"davis": {"all": (0.3516, 0.8385), "uncensored": (0.6875, 0.7138)},
        "kiba": {"all": (0.2788, 0.7871)}}

runs = {}
for path in sorted(Path(ROOT).glob("**/metrics.json")):
    payload = json.loads(path.read_text())
    runs.setdefault(payload["dataset"], []).append(payload)

for dataset, payloads in runs.items():
    payloads.sort(key=lambda row: row["seed"])
    print()
    print("===== " + dataset + " frozen ESM-2, " + str(len(payloads)) + " seed(s) =====")
    for payload in payloads:
        line = "  seed%d best_ep %3d" % (payload["seed"], payload["best_epoch"])
        for view in ("all", "uncensored"):
            if view in payload["test"]:
                line += "  %s MSE %.4f CI %.4f" % (view[:5], payload["test"][view]["mse"],
                                                   payload["test"][view]["ci"])
        print(line)
    for view in ("all", "uncensored"):
        if view not in payloads[0]["test"]:
            continue
        esm_mse = [p["test"][view]["mse"] for p in payloads]
        esm_ci = [p["test"][view]["ci"] for p in payloads]
        cnn_mse, cnn_ci = CNN[dataset][view]["mse"], CNN[dataset][view]["ci"]
        print("  %-10s ESM  MSE %.4f  CI %.4f   (n=%d seeds)"
              % (view, st.mean(esm_mse), st.mean(esm_ci), len(esm_mse)))
        print("  %-10s CNN  MSE %.4f  CI %.4f" % (view, st.mean(cnn_mse), st.mean(cnn_ci)))
        print("  %-10s kmer MSE %.4f  CI %.4f" % (view, KMER[dataset][view][0], KMER[dataset][view][1]))
        print("  %-10s ESM - CNN  dMSE %+.4f  dCI %+.4f"
              % (view, st.mean(esm_mse) - st.mean(cnn_mse), st.mean(esm_ci) - st.mean(cnn_ci)))
        # Paired by seed: both arms used the same seed for weights and the same split.
        if len(esm_mse) == len(cnn_mse):
            deltas = [e - c for e, c in zip(esm_mse, cnn_mse)]
            print("  %-10s per-seed dMSE %s" % (view, [round(d, 4) for d in deltas]))
            if len(deltas) > 1 and st.stdev(deltas) > 0:
                t = st.mean(deltas) / (st.stdev(deltas) / len(deltas) ** 0.5)
                print("  %-10s paired t(%d) = %+.2f  (read the per-seed signs above first)"
                      % (view, len(deltas) - 1, t))


In [ ]:
# The two views the pooled numbers hide: can the model order an unseen target against another
# unseen target, and does it predict anything other than a constant per drug?
#
# Definitions are the SC4020 session's, and the same ones used for the CNN arms:
#   within-ligand CI     concordance inside each test drug's own measured test pairs.
#   predicted spread     for each test drug, predict it against EVERY cold test target whether
#                        or not the pair was measured, take the sd, average over drugs. Exactly
#                        0 for a model that cannot separate unseen targets.
# Both are restricted to the matched ligand set: drugs with at least two measured test pairs and
# a non-zero measured spread. That is the only basis on which the spread ratio means anything --
# 2 of 68 DAVIS and 429 of 2025 KIBA drugs have a measured spread of exactly zero.
sys.path.insert(0, str(project))
import numpy as np
import pandas as pd
from src.config import Config
from src.data import load_esm_features, processed_csv, resolve_split_column
from src.model import DeepDTA

DATA = project / "data" / "processed"
CNN_WITHIN = {"davis": {"ci": 0.6789, "spread": 0.3679, "ratio": 0.576, "measured": 0.6391},
              "kiba": {"ci": 0.6363, "spread": 0.2960, "ratio": 0.514, "measured": 0.5762}}
KMER_WITHIN = {"davis": {"ci": 0.7494, "ratio": 0.771}, "kiba": {"ci": 0.6467, "ratio": 0.401}}
device = torch.device("cuda")


def pairwise_ci(y, prediction):
    """Return one drug's concordance index, prediction ties counted as half."""
    y, prediction = np.asarray(y, dtype=float), np.asarray(prediction, dtype=float)
    better = y[:, None] > y[None, :]
    if not better.any():
        return None
    gap = prediction[:, None] - prediction[None, :]
    return float(((gap[better] > 0).sum() + 0.5 * (gap[better] == 0).sum()) / better.sum())


def batched(model, drug_tokens, protein_features, drug_rows, protein_rows, batch=1024):
    """Predict for aligned drug/target row-index arrays."""
    out = []
    with torch.no_grad():
        for start in range(0, len(drug_rows), batch):
            d = drug_tokens[drug_rows[start:start + batch]].to(device)
            p = protein_features[protein_rows[start:start + batch]].to(device)
            out.append(model(d, p).detach().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


summary = {}
for path in sorted(Path(ROOT).glob("**/metrics.json")):
    run_dir = path.parent
    saved = json.loads((run_dir / "config.json").read_text())
    config = Config(**{k: v for k, v in saved.items() if k in Config.__dataclass_fields__})
    dataset, split = config.dataset, config.split
    model = DeepDTA(saved["vocab_sizes"]["smiles"], saved["vocab_sizes"]["protein"], config).to(device)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device, weights_only=True))
    model.eval()

    pairs = pd.read_csv(processed_csv(DATA, dataset + "_pairs"),
                        dtype={"ligand_id": str, "target_id": str})
    ligands = pd.read_csv(processed_csv(DATA, dataset + "_ligands"), usecols=["ligand_id"],
                          dtype={"ligand_id": str})
    targets = pd.read_csv(processed_csv(DATA, dataset + "_targets"), usecols=["target_id"],
                          dtype={"target_id": str})
    ligand_row = dict(zip(ligands.ligand_id, range(len(ligands))))
    target_row = dict(zip(targets.target_id, range(len(targets))))
    test = pairs.loc[resolve_split_column(pairs, DATA, dataset, split) == "test"].copy()

    with np.load(DATA / (dataset + "_tokens.npz"), allow_pickle=False) as arrays:
        drug_tokens = torch.tensor(arrays["drug_tokens"], dtype=torch.long)
    protein_features = load_esm_features(DATA, dataset, len(targets), config.esm_dim)

    test["pred"] = batched(model, drug_tokens, protein_features,
                           test.ligand_id.map(ligand_row).to_numpy(),
                           test.target_id.map(target_row).to_numpy())
    measured = test.groupby("ligand_id").score.std(ddof=1)
    matched = sorted(measured[(measured > 0) & measured.notna()].index)
    cis = [v for drug in matched
           if (v := pairwise_ci(test.loc[test.ligand_id == drug, "score"],
                                test.loc[test.ligand_id == drug, "pred"])) is not None]

    test_targets = sorted(test.target_id.unique())
    grid_drugs = np.repeat([ligand_row[d] for d in matched], len(test_targets))
    grid_prots = np.tile([target_row[t] for t in test_targets], len(matched))
    grid = batched(model, drug_tokens, protein_features, grid_drugs, grid_prots)
    spreads = grid.reshape(len(matched), len(test_targets)).std(axis=1, ddof=1)
    measured_mean = float(measured.loc[matched].mean())

    summary.setdefault(dataset, []).append((config.seed, float(np.mean(cis)),
                                            float(spreads.mean()), measured_mean,
                                            int((spreads < 1e-9).sum()), len(matched)))
    print("esm %-6s seed%d  within-ligand CI %.4f  predicted spread %.4f  "
          "measured %.4f  ratio %.3f  drugs at predicted spread 0: %d of %d"
          % (dataset, config.seed, np.mean(cis), spreads.mean(), measured_mean,
             spreads.mean() / measured_mean, (spreads < 1e-9).sum(), len(matched)), flush=True)

for dataset, rows in summary.items():
    ci = sum(r[1] for r in rows) / len(rows)
    spread = sum(r[2] for r in rows) / len(rows)
    measured_mean = rows[0][3]
    print()
    print("===== " + dataset + " within-ligand means over " + str(len(rows)) + " seed(s) =====")
    print("  ESM   within-ligand CI %.4f  spread %.4f  spread/measured %.3f"
          % (ci, spread, spread / measured_mean))
    print("  CNN   within-ligand CI %.4f  spread %.4f  spread/measured %.3f"
          % (CNN_WITHIN[dataset]["ci"], CNN_WITHIN[dataset]["spread"], CNN_WITHIN[dataset]["ratio"]))
    print("  kmer  within-ligand CI %.4f  spread/measured %.3f"
          % (KMER_WITHIN[dataset]["ci"], KMER_WITHIN[dataset]["ratio"]))
    print("  ESM - CNN  dCI %+.4f  dspread %+.4f"
          % (ci - CNN_WITHIN[dataset]["ci"], spread - CNN_WITHIN[dataset]["spread"]))
    print("  If dCI is positive the ESM branch orders unseen targets better, which is the claim.")
    print("  If dCI is flat while test MSE improved, ESM bought calibration, not separation.")
